# igPC applied to Van der Pol data — Tutorial

This notebook shows how to apply the igPC identification method to a signal
generated by the **full stochastic Van der Pol oscillator** (Eq. 2), which is
closer to what one would have in practice from a real thermoacoustic measurement.

The key extra step compared to the Langevin tutorial is the **preprocessing
pipeline**: raw pressure signal → band-pass filter → Hilbert transform → amplitude
envelope A(t). This is exactly what a practitioner would do with measured data.

---
**Outline**
1. Simulate the stochastic Van der Pol ODE
2. Envelope extraction: state-space (ground truth) vs. Hilbert transform (realistic)
3. Mean-square displacement and diffusion coefficient *c*
4. Two-level correction for non-Gaussian envelopes
5. Parameter identification via Eqs. 10–11
6. Summary and comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# ooigpc modules
from ooigpc.solvers.van_der_pol import simulate_van_der_pol, amplitude_envelope
from ooigpc.signal import bandpass_butter, hilbert_envelope
from ooigpc.diffusion import compute_msd, fit_msd
from ooigpc.correction import apply_correction
from ooigpc.igpc import identify

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

---
## 1 — Simulate the stochastic Van der Pol ODE

The stochastic Van der Pol equation (Eq. 2) is:

$$
\ddot{\eta} - 2\nu\,\dot{\eta} + \kappa\,\eta^2\,\dot{\eta} + \omega_0^2\,\eta = \Xi(t)
$$

where $\Xi(t)$ is Gaussian white noise with intensity $\Gamma$.
The cubic term $-\kappa\eta^2\dot{\eta}$ provides the amplitude saturation that
produces the limit cycle at $A_\text{eq} = \sqrt{8\nu/\kappa}$ and, after
slow-amplitude averaging, yields the Langevin drift $\nu A - \tfrac{\kappa}{8}A^3$
(Eq. 3).
The same physical parameters as the Langevin tutorial are used so that
results can be compared directly.

In [ ]:
# Physical parameters
omega  = 2 * np.pi * 120   # ω₀ = 2π × 120 Hz
f0     = 120.0             # fundamental frequency (Hz)
nu     = 16.0              # linear growth rate ν  (s⁻¹)
kappa  = 3.4               # nonlinear coefficient κ
Gamma  = 1e7               # noise intensity Γ  (Pa²·s⁻³)
dt     = 5e-5              # time step Δt (s)
fs     = 1.0 / dt          # sampling frequency (Hz)
N      = 2_000_000         # number of steps

time, eta, eta_dot = simulate_van_der_pol(
    nu=nu, kappa=kappa, Gamma=Gamma, omega=omega,
    N=N, dt=dt, seed=0
)

print(f"Signal duration : {N*dt:.1f} s  ({N*dt*f0:.0f} oscillation cycles)")
print(f"Sampling rate   : {fs:.0f} Hz")
print(f"eta.shape       : {eta.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Short excerpt of the raw signal
n_show = int(0.1 / dt)   # 0.1 s excerpt
axes[0].plot(time[:n_show], eta[:n_show], lw=0.6, color='steelblue')
axes[0].set(xlabel='Time (s)', ylabel='η(t)', title='Raw Van der Pol signal (excerpt)')

# Power spectral density
freqs, psd = welch(eta, fs=fs, nperseg=8192)
axes[1].semilogy(freqs, psd, lw=0.8, color='steelblue')
axes[1].axvline(f0, color='r', ls='--', lw=1.5, label=f'f₀ = {f0:.0f} Hz')
axes[1].set(xlabel='Frequency (Hz)', ylabel='PSD', title='Power spectral density',
            xlim=[0, 600])
axes[1].legend()

plt.tight_layout()
plt.show()

\
---
## 2 — Envelope extraction

Two methods are compared:

**State-space envelope** (ground truth, only available in simulation):
$$A_\text{ref}(t) = \sqrt{\eta(t)^2 + \left(\frac{\dot{\eta}(t)}{\omega_0}\right)^2}$$

This is exact for a pure harmonic at $\omega_0$ and requires no filtering because
both $\eta$ and $\dot{\eta}$ are available from the ODE solver.

**Hilbert-transform envelope** (the realistic workflow for measured data):
1. **Band-pass filter** $\eta(t)$ around $f_0 \pm \Delta f$ to isolate the mode
2. **Hilbert transform** → analytic signal $\eta_a(t) = \eta_\text{bp}(t) + i\,\mathcal{H}\{\eta_\text{bp}(t)\}$
3. **Envelope**: $A_\text{Hilbert}(t) = |\eta_a(t)|$

The igPC method will be applied to $A_\text{Hilbert}$, which is what a
practitioner would compute from a measured pressure signal.


In [ ]:
# ── Ground-truth envelope (state-space) ──────────────────────────────────────
A_ref = amplitude_envelope(eta, eta_dot, omega)

# ── Realistic envelope (bandpass + Hilbert) ───────────────────────────────────
bw     = 20.0   # half-bandwidth ±20 Hz around f₀
eta_bp = bandpass_butter(eta, fs=fs, f_low=f0 - bw, f_high=f0 + bw, order=4)
A_hil  = hilbert_envelope(eta_bp)

# ── Comparison plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

n_cmp = int(0.5 / dt)   # 0.5 s comparison window
t_cmp = time[:n_cmp]

axes[0].plot(t_cmp, A_ref[:n_cmp], lw=1.2, label='State-space (ref)', color='steelblue')
axes[0].plot(t_cmp, A_hil[:n_cmp], lw=1.2, label='Hilbert', color='tomato', alpha=0.8)
axes[0].set(xlabel='Time (s)', ylabel='A(t)', title='Envelope comparison (excerpt)')
axes[0].legend()

# Scatter: both envelopes on same samples (after transient)
i0 = 5000
axes[1].scatter(A_ref[i0::50], A_hil[i0::50], s=1, alpha=0.3, color='steelblue')
lim = [0, max(A_ref[i0:].max(), A_hil[i0:].max()) * 1.05]
axes[1].plot(lim, lim, 'r--', lw=1.5, label='ideal')
axes[1].set(xlabel='A_ref', ylabel='A_Hilbert', title='Scatter: ref vs. Hilbert',
            xlim=lim, ylim=lim, aspect='equal')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean A_ref    = {A_ref[i0:].mean():.4f}")
print(f"Mean A_Hilbert = {A_hil[i0:].mean():.4f}")

\
---
## 3 — Mean-square displacement and diffusion coefficient *c*

From here the pipeline is identical to the Langevin tutorial — the only
difference is that the input envelope is $A_\text{Hilbert}$ rather than a
directly simulated Langevin process.

MSD model (Eq. 40):
$$\text{MSD}_{\text{mod}}(\tau) = 2\,C(0)\,[1 - e^{-\lambda\tau}]$$

Diffusion coefficient (Eq. 14):
$$c = C(0)\cdot\lambda$$


In [ ]:
\
tau_values = np.arange(2.5e-4, 0.201, 2.5e-4)

taus, msd = compute_msd(A_hil, tau_values, dt_sim=dt)
A_hat, T_hat, msd_fit = fit_msd(taus, msd)
c_recov = A_hat / (2 * T_hat)

# True diffusion coefficient
c_true = Gamma / (4 * omega**2)
print(f"C(0)    = {A_hat:.4f}")
print(f"λ       = {1/T_hat:.4f} s⁻¹")
print(f"c       = {c_recov:.4f}  (true: {c_true:.4f},  error: {abs(c_recov-c_true)/c_true*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(taus, msd, 'rx', ms=3, label='MSD data')
axes[0].plot(taus, msd_fit, 'b-', lw=2,
             label=r'$2\,C(0)\,[1-e^{-\lambda\tau}]$  [Eq. 40]')
axes[0].set(xlabel=r'$\tau$ (s)', ylabel='MSD', title='MSD fit (Eq. 40)')
axes[0].legend()

axes[1].plot(taus, msd / (2*taus), 'rx', ms=3)
axes[1].plot(taus, msd_fit / (2*taus), 'b-', lw=2)
axes[1].axhline(c_recov, color='k', ls='--', label=f'c = {c_recov:.4f}  [Eq. 14]')
axes[1].axhline(c_true,  color='r', ls=':',  label=f'c true = {c_true:.4f}')
axes[1].set(xlabel=r'$\tau$ (s)', ylabel='MSD / (2τ)', title='Extrapolation τ→0  (Eq. 14)')
axes[1].legend()

plt.tight_layout()
plt.show()


\
---
## 4 — Two-level correction for non-Gaussian envelopes

Same correction as in the Langevin tutorial — marginally stable regime uses
Eqs. 30–31 and 33; linearly unstable regime uses Eq. 20.


In [ ]:
amp_positive = A_hil[5000:][A_hil[5000:] > 0]
corr = apply_correction(amp_positive)

print(f"μ             = {corr.mu:.4f}")
print(f"std_raw       = {corr.std_raw:.4f}")
print(f"σ (corrected) = {corr.sigma:.4f}")
print(f"Region = {corr.region}  ({'marginally stable' if corr.region==1 else 'linearly unstable'})")

\
---
## 5 — Parameter identification (Eqs. 10–11)

$$
\nu = \frac{9\sigma^4 + 2\mu^2\sigma^2 - \mu^4}{2\mu^2\sigma^2(\sigma^2 - \mu^2)} \cdot c
\tag{10}
$$

$$
\frac{\kappa}{8} = \frac{3\sigma^2 - \mu^2}{2\mu^2\sigma^2(\sigma^2 - \mu^2)} \cdot c
\tag{11}
$$


In [ ]:
ident = identify(amp_positive, c=c_recov, omega=omega)

err_nu    = abs(ident.nu    - nu)    / abs(nu)    * 100
err_kappa = abs(ident.kappa - kappa) / abs(kappa) * 100
err_Gamma = abs(ident.Gamma - Gamma) / abs(Gamma) * 100

print("Identification results (from Van der Pol + Hilbert pipeline):")
print(f"  ν     = {ident.nu:.2f}  s⁻¹   (true: {nu:.1f},   error: {err_nu:.1f}%)")
print(f"  κ     = {ident.kappa:.3f}        (true: {kappa:.2f},  error: {err_kappa:.1f}%)")
print(f"  Γ     = {ident.Gamma:.3e}  (true: {Gamma:.1e}, error: {err_Gamma:.1f}%)")

---
## 6 — Summary and comparison

The table below collects the identified parameters from both the
state-space ground-truth envelope and the Hilbert-transform envelope,
showing that the full preprocessing pipeline introduces negligible extra error.

In [ ]:
\
from ooigpc.diffusion import compute_msd, fit_msd
from ooigpc.igpc import identify

# Identify also from the ground-truth envelope (for comparison)
A_ref_pos = A_ref[5000:][A_ref[5000:] > 0]
taus_r, msd_r = compute_msd(A_ref_pos, tau_values, dt_sim=dt)
A_hat_r, T_hat_r, _ = fit_msd(taus_r, msd_r)
c_ref = A_hat_r / (2 * T_hat_r)
ident_ref = identify(A_ref_pos, c=c_ref, omega=omega)

rows = [
    ("True",               nu,             kappa,              Gamma),
    ("State-space env.",   ident_ref.nu,   ident_ref.kappa,    ident_ref.Gamma),
    ("Hilbert env.",       ident.nu,       ident.kappa,        ident.Gamma),
]

print(f"{'Method':<22}  {'ν (s⁻¹)':>10}  {'κ':>8}  {'Γ':>12}")
print("-" * 58)
for label, nv, kp, gm in rows:
    print(f"{label:<22}  {nv:>10.2f}  {kp:>8.3f}  {gm:>12.3e}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
methods = ['True', 'State-space', 'Hilbert']
nu_vals    = [nu,    ident_ref.nu,    ident.nu]
kappa_vals = [kappa, ident_ref.kappa, ident.kappa]
Gamma_vals = [Gamma, ident_ref.Gamma, ident.Gamma]

colors = ['#2c7bb6', '#abd9e9', '#fdae61']
for ax, vals, label in zip(axes,
                            [nu_vals, kappa_vals, Gamma_vals],
                            ['ν  (s⁻¹)', 'κ', 'Γ  (Pa²·s⁻³)']):
    bars = ax.bar(methods, vals, color=colors, edgecolor='k', linewidth=0.5)
    ax.axhline(vals[0], color='k', ls='--', lw=1.2, label='True')
    ax.set(ylabel=label, title=label)
    ax.legend(fontsize=9)

plt.suptitle('igPC identification — Van der Pol + Hilbert pipeline', y=1.02)
plt.tight_layout()
plt.show()
